Fetches user-sourced data from MongoDB and uses it to retrain the ML classifier

In [23]:
import pandas as pd
from PIL import Image
import numpy as np

import tensorflow as tf
from tensorflow.keras import layers, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import os
from dotenv import load_dotenv
from pymongo import MongoClient

import base64
import io

In [24]:
load_dotenv("../src/credentials/.env")
db_url = os.getenv("MONGO_CONNECTION_STRING")

client = MongoClient(db_url)
db = client["classifier"]
collection = db["feedback"]

In [25]:
total = collection.count_documents({})
num_correct = collection.count_documents({"was_correct" : True})
num_incorrect = collection.count_documents({"was_correct" : False})

print(f"Out of {total} responses, {num_correct} were correct, and {num_incorrect} were incorrect")

Out of 91 responses, 49 were correct, and 42 were incorrect


In [ ]:
# Make a list of all user-drawn images
resized_img = []
y = []

data = collection.find({})
for entry in data:
    imageData = entry["image_b64"]
    bytes = base64.b64decode(imageData.split(",")[1])

    img = Image.open(io.BytesIO(bytes))
    img = img.resize((64, 64))
    
    resized_img.append(img)
    y.append(entry["true_label"])

In [ ]:
# Combine with previous dataset
# Extract the sample paths and labels
csv = pd.read_csv('../data/english.csv')
paths = csv['image'].tolist()
y.extend(csv['label'].tolist())

# Get the 62 classes
classes = csv['label'].unique().tolist()

# Convert class labels to integer
conversion = {}

cur = 0
for label in classes:
    conversion[label] = cur
    cur += 1

for i in range(len(y)):
    y[i] = conversion[f"{y[i]}"]

for image in paths:
    img = Image.open(f"../data/{image}")
    img = img.resize((64, 64))
    
    resized_img.append(img)

In [ ]:
# Preprocess and stack
X = []
for img in resized_img:
    img = img.convert("L")

    img_arr = np.array(img)
    img_arr = img_arr / 255.0

    img_arr = np.expand_dims(img_arr, axis=-1)

    X.append(img_arr)

X = np.array(X, dtype="float32")
y = np.array(y, dtype="int32")

In [29]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [30]:
# Try 1NN first
X_flat = X.reshape(X.shape[0], -1)

X_flat_train, X_flat_test, y_train, y_test = train_test_split(X_flat, y, test_size=0.2, random_state=42)

def one_nn(X_flat_train, y_train, X_flat_test):
    y_pred = []
    for test in X_flat_test:
        dist = np.sum((X_flat_train - test) ** 2, axis=1)
        y_pred.append(y_train[np.argmin(dist)])
    
    return np.array(y_pred)

y_pred = one_nn(X_flat_train, y_train, X_flat_test)

# Evaluate
print(classification_report(y_test, y_pred, target_names=classes))

              precision    recall  f1-score   support

           0       0.20      0.30      0.24        10
           1       0.38      0.40      0.39        20
           2       1.00      0.40      0.57        15
           3       0.50      0.33      0.40        18
           4       0.56      0.38      0.45        13
           5       0.29      0.17      0.21        12
           6       0.58      0.47      0.52        15
           7       0.69      0.64      0.67        14
           8       0.20      0.17      0.18        12
           9       0.44      0.40      0.42        10
           A       0.74      1.00      0.85        14
           B       0.00      0.00      0.00        10
           C       0.56      0.77      0.65        13
           D       0.75      0.25      0.38        12
           E       0.47      0.54      0.50        13
           F       0.31      0.44      0.36         9
           G       1.00      0.44      0.62         9
           H       0.67    

In [31]:
# Create CNN
def init_model():
    model = models.Sequential()

    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 1)))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(128, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(62, activation="softmax"))

    return model

In [32]:
model = init_model()
model.summary()

C:\Users\Tiger\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 62, 62, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 62)             │         7,998 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 690,622 (2.63 MB)

 Trainable params: 690,622 (2.63 MB)

 Non-trainable params: 0 (0.00 B)

In [33]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=10,
                    validation_data=(X_test, y_test))

Epoch 1/10


C:\Users\Tiger\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\backend\tensorflow\nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


88/88 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.0486 - loss: 3.9846 - val_accuracy: 0.1469 - val_loss: 3.5158
Epoch 2/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.2525 - loss: 2.8819 - val_accuracy: 0.4194 - val_loss: 2.2555
Epoch 3/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4343 - loss: 2.0032 - val_accuracy: 0.5606 - val_loss: 1.6779
Epoch 4/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5821 - loss: 1.4611 - val_accuracy: 0.5963 - val_loss: 1.4001
Epoch 5/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6586 - loss: 1.1501 - val_accuracy: 0.6690 - val_loss: 1.1580
Epoch 6/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7096 - loss: 0.9457 - val_accuracy: 0.6776 - val_loss: 1.1381
Epoch 7/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7554 - loss: 0.7842 - val_accuracy: 0.6947 - val_loss: 1.0694
Epoch 8/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7932 - loss: 0.6618 - val_accuracy: 0.7161 - val_loss: 1.

In [34]:
# Evaluation
print(model.evaluate(X_test, y_test))

y_prob = model.predict(X_test)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_test, y_pred, target_names=classes))

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7204 - loss: 1.0457
[1.0456815958023071, 0.7203994393348694]
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
              precision    recall  f1-score   support

           0       0.35      0.60      0.44        10
           1       0.67      0.20      0.31        20
           2       0.83      0.67      0.74        15
           3       0.86      0.67      0.75        18
           4       0.82      0.69      0.75        13
           5       0.92      0.92      0.92        12
           6       0.67      0.93      0.78        15
           7       0.80      0.86      0.83        14
           8       0.53      0.83      0.65        12
           9       0.80      0.40      0.53        10
           A       1.00      0.93      0.96        14
           B       0.75      0.60      0.67        10
           C       0.67      0.77      0.71        13
           D       0.78      0.58      0.67        12
           E       0.92      0.85   

In [35]:
# Exporting model
model.save("../src/models/char_cnn_feedback.keras")

Not much improvement since we only added 91 new entries.